# Pedroni Panel SVAR Analysis - DeFi Liquidations

This notebook runs Panel SVAR analysis on the relationship between:
- **Liquidation**: log-transformed USD value of collateral seized
- **Utilization**: Leverage ratio (borrowed/supplied)
- **Volatility**: Rolling std of collateral basket returns

Using Pedroni (2013) Panel SVAR methodology across 31 qualified DeFi protocols (T=592, start 2024-07-01).

In [2]:
import sys
from pathlib import Path

# Add pedroni_svar code to path
sys.path.insert(0, str(Path('../code/pedroni_svar')))

from SVAR import *
from panelSVAR import *
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

print("Libraries loaded successfully")

ModuleNotFoundError: No module named 'statsmodels'

## 1. Load and Inspect Data

In [ ]:
# Load panel data
panel = pd.read_parquet('../data/analysis/panel_svar_data_qualified.parquet')

print(f"Panel dimensions:")
print(f"  CSUs: {panel['csu'].nunique()}")
print(f"  Date range: {panel['date'].min()} to {panel['date'].max()}")
print(f"  Total observations: {len(panel):,}")
print()
print(f"Variable coverage:")
for col in ['liquidation', 'utilization', 'volatility']:
    coverage = panel[col].notna().mean() * 100
    print(f"  {col}: {coverage:.1f}%")

print(f"\nQualified CSUs ({panel['csu'].nunique()}):")
for csu in sorted(panel['csu'].unique()):
    print(f"  {csu}")

In [ ]:
# Preview data
panel.head(10)

## 2. Configure Panel SVAR

In [ ]:
# Configuration
plot = True
savefig_path = "../data/analysis/svar_figures/"
excel_path = "../data/analysis/panel_svar_data_qualified.xlsx"
excel_sheet_name = "panel_data"

# Create required directories
Path("../output").mkdir(parents=True, exist_ok=True)
Path(savefig_path).mkdir(parents=True, exist_ok=True)

# Variables: [input_form, output_form]
# 0 = stationary, 1 = unit root
variables = {
    'utilization': [0, 0],  # Stationary (most exogenous)
    'liquidation': [0, 0],  # Stationary (middle)
    'volatility': [0, 0],   # Stationary (most endogenous)
}

# IMPORTANT: Order reflects causal structure
# Utilization → Liquidation → Volatility
variable_order = ['utilization', 'liquidation', 'volatility']
shocks = ['Utilization Shock', 'Liquidation Shock', 'Volatility Shock']

# Panel structure
td_col = ["date"]
member_col = "csu"

# Long-run restrictions (Cholesky ordering)
# Causal chain: Utilization causes Liquidation causes Volatility
lr_constraint = np.array([
    ['.', '0', '0'],  # Utilization: only utilization shock has long-run effect
    ['.', '.', '0'],  # Liquidation: utilization & liquidation shocks have effects
    ['.', '.', '.']   # Volatility: all shocks have long-run effects
])

# This means:
# - Utilization shock → affects utilization, liquidation, volatility (most exogenous)
# - Liquidation shock → affects liquidation, volatility (not utilization)
# - Volatility shock → affects volatility only (most endogenous)

# Sign restrictions (required: one per column)
lr_sign = np.array([
    ['+', '.', '.'],  # Utilization shock → positive utilization
    ['.', '+', '.'],  # Liquidation shock → positive liquidation
    ['.', '.', '+']   # Volatility shock → positive volatility
])

sr_constraint = np.array([])  # No short-run constraints

# Analysis parameters
maxlags = 7       # Test up to 7 lags
nsteps = 20       # 20-step impulse responses
lagmethod = 'aic' # Use AIC for lag selection

# Bootstrap
bootstrap = True
ndraws = 1000
signif = 0.05

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"\nCausal ordering: Utilization -> Liquidation -> Volatility")
print(f"Variables: {variable_order}")
print(f"Max lags: {maxlags}, Steps: {nsteps}")
print(f"Bootstrap: {bootstrap} ({ndraws} draws)")
print(f"\nLong-run constraints (Cholesky):")
print(lr_constraint)
print(f"\nSign restrictions:")
print(lr_sign)
print("="*70)

## 3. Run Panel SVAR Analysis

In [ ]:
# Create output directory
Path(savefig_path).mkdir(parents=True, exist_ok=True)

# Create input object
panel_input = VAR_input(
    variables=variables,
    variable_order=variable_order,
    shocks=shocks,
    td_col=td_col,
    member_col=member_col,
    M=None,
    sr_constraint=sr_constraint,
    lr_constraint=lr_constraint,
    lr_sign=lr_sign,  # FIXED: Changed from sr_sign to lr_sign
    maxlags=maxlags,
    nsteps=nsteps,
    lagmethod=lagmethod,
    bootstrap=bootstrap,
    ndraws=ndraws,
    signif=signif,
    excel_path=excel_path,
    excel_sheet_name=excel_sheet_name,
    df=pd.DataFrame(),
    plot=plot,
    savefig_path=savefig_path
)

print("Running Pedroni Panel SVAR...")
print("This may take several minutes (5-10 min for 1000 bootstrap draws)...\n")

# Run analysis
output = panelSVAR(panel_input)

print("\n" + "="*70)
print("✓ Analysis complete!")
print("="*70)
print(f"Results saved to: ../output/")
print(f"Figures will be generated in next cell")

## 4. View Results

In [ ]:
# Display impulse responses
if hasattr(output, 'ir'):
    print("Impulse Responses:")
    print(output.ir)

In [ ]:
# List generated figures
import os

fig_dir = Path(savefig_path)
if fig_dir.exists():
    figures = list(fig_dir.glob('*.png')) + list(fig_dir.glob('*.pdf'))
    print(f"Generated {len(figures)} figures:")
    for fig in sorted(figures):
        print(f"  {fig.name}")
else:
    print("No figures directory found")

In [ ]:
# Generate IRF plots from results
import matplotlib.pyplot as plt
import seaborn as sns

# Read common shock results
df_common = pd.read_excel('../output/ind-IRs-to-common-shocks.xlsx')

# Create 3x3 grid of IRF plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Impulse Responses to Common Shocks\nPanel SVAR - DeFi Liquidations (31 CSUs, T=592)\nCausal Order: Utilization -> Liquidation -> Volatility',
             fontsize=14, fontweight='bold')

# Updated variable order to match causal structure
variables = ['utilization', 'liquidation', 'volatility']

for i in range(3):
    for j in range(3):
        ax = axes[i, j]
        
        # Get IRF data (columns are like IR11_0, IR11_1, ..., IR11_20 for var1->shock1)
        col_prefix = f'IR{i+1}{j+1}_'
        cols = [col for col in df_common.columns if col.startswith(col_prefix)]
        
        if cols:
            # Average across CSUs
            irf_data = df_common[cols].mean(axis=0).values
            steps = range(len(irf_data))
            
            # Plot
            ax.plot(steps, irf_data, 'b-', linewidth=2.5)
            ax.axhline(0, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
            ax.fill_between(steps, 0, irf_data, alpha=0.2)
            
            # Labels with clearer notation
            response_var = variables[i].capitalize()
            shock_var = variables[j].capitalize()
            ax.set_title(f"{response_var} Response to {shock_var} Shock", 
                        fontsize=10, fontweight='bold')
            if i == 2:
                ax.set_xlabel('Steps')
            if j == 0:
                ax.set_ylabel('Response')
            ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(savefig_path + 'common_shock_irfs.png', dpi=300, bbox_inches='tight')
plt.savefig(savefig_path + 'common_shock_irfs.pdf', bbox_inches='tight')
print("Saved IRF plots to", savefig_path)
plt.show()